In [0]:
# ============================================================
# NOTEBOOK 4: FAIRNESS AUDIT + GOLD TABLE
# Team AryaBit | HackBricks 2026
# Purpose: Audit model for bias + produce final Gold output
# Metrics: Demographic Parity + Equal Opportunity
# ============================================================

import logging
import pandas as pd
import numpy as np
import mlflow
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    round as spark_round
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, TimestampType
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.Fairness")

# ── CONFIG ───────────────────────────────────────────────────
CATALOG             = "workspace"
SCHEMA              = "default"
SILVER_TABLE        = f"{CATALOG}.{SCHEMA}.silver_student_dropout"
GOLD_TABLE          = f"{CATALOG}.{SCHEMA}.gold_at_risk_students"
FAIRNESS_TABLE      = f"{CATALOG}.{SCHEMA}.gold_fairness_audit"
EXPERIMENT_NAME     = "/AryaBit_HackBricks_2026"

FEATURE_COLS = [
    "grade_delta", "absenteeism_trend", "financial_stress_index",
    "overall_approval_rate", "avg_grade_overall",
    "curricular_units_1st_sem_grade", "curricular_units_2nd_sem_grade",
    "curricular_units_1st_sem_approved", "curricular_units_2nd_sem_approved",
    "curricular_units_1st_sem_enrolled", "curricular_units_2nd_sem_enrolled",
    "admission_grade", "age_at_enrollment", "debtor",
    "tuition_fees_up_to_date", "scholarship_holder", "displaced",
]
TARGET_COL = "dropout_label"

# ── STEP 1: LOAD MODEL + GENERATE PREDICTIONS ────────────────
def load_and_predict():
    logger.info("Loading trained model from MLflow...")
    
    # Get the best model from experiment runs
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    
    if not experiment:
        raise Exception(f"Experiment '{EXPERIMENT_NAME}' not found. Run Notebook 3 first.")
    
    # Get the best run (highest ROC-AUC)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.mlflow.runName = 'RandomForest_AryaBit'",
        order_by=["metrics.roc_auc DESC"],
        max_results=1
    )
    
    if len(runs) == 0:
        raise Exception(f"No RandomForest runs found in experiment. Run Notebook 3 first.")
    
    best_run = runs.iloc[0]
    run_id = best_run['run_id']
    roc_auc = best_run['metrics.roc_auc']
    
    logger.info(f"Loading model from run {run_id} (ROC-AUC: {roc_auc:.4f})")
    
    # Load model from the run
    model_uri = f"runs:/{run_id}/random_forest_model"
    rf_model = mlflow.sklearn.load_model(model_uri)
    logger.info("Model loaded successfully ✅")
    
    logger.info("Loading Silver table and generating predictions...")
    silver_df = spark.table(SILVER_TABLE)
    pandas_df = silver_df.toPandas()
    
    X = pandas_df[FEATURE_COLS]
    y_true = pandas_df[TARGET_COL]
    
    # Generate predictions
    y_pred = rf_model.predict(X)
    y_prob = rf_model.predict_proba(X)[:, 1]
    
    pandas_df["risk_score"]    = np.round(y_prob, 4)
    pandas_df["predicted_dropout"] = y_pred
    
    logger.info(f"Predictions generated: {len(pandas_df)} records")
    return pandas_df

# ── STEP 2: FAIRNESS AUDIT ───────────────────────────────────
def compute_fairness_metrics(df):
    """
    Compute Demographic Parity + Equal Opportunity
    across Gender and Financial groups.
    
    Gender:    0=Female, 1=Male
    Debtor:    0=No, 1=Yes (financial stress proxy)
    Scholarship: 0=No, 1=Yes
    """
    logger.info("Computing fairness metrics...")
    
    audit_results = []
    
    # ── FAIRNESS ACROSS GENDER ────────────────────────────────
    for gender_val, gender_label in [(0, "Female"), (1, "Male")]:
        group = df[df["gender"] == gender_val]
        
        if len(group) == 0:
            continue
            
        # Demographic Parity: % predicted as dropout
        dem_parity = group["predicted_dropout"].mean()
        
        # Equal Opportunity: recall among actual dropouts
        actual_dropouts = group[group[TARGET_COL] == 1]
        eq_opportunity = (
            actual_dropouts["predicted_dropout"].mean()
            if len(actual_dropouts) > 0 else 0
        )
        
        # False Positive Rate: flagged but not actual dropout
        actual_non_dropouts = group[group[TARGET_COL] == 0]
        fpr = (
            actual_non_dropouts["predicted_dropout"].mean()
            if len(actual_non_dropouts) > 0 else 0
        )
        
        audit_results.append({
            "group_type":        "Gender",
            "group_label":       gender_label,
            "group_size":        len(group),
            "actual_dropout_rate": round(group[TARGET_COL].mean(), 4),
            "predicted_dropout_rate": round(dem_parity, 4),
            "demographic_parity": round(dem_parity, 4),
            "equal_opportunity": round(eq_opportunity, 4),
            "false_positive_rate": round(fpr, 4),
        })
    
    # ── FAIRNESS ACROSS FINANCIAL STATUS (DEBTOR) ─────────────
    for debtor_val, debtor_label in [(0, "Non-Debtor"), (1, "Debtor")]:
        group = df[df["debtor"] == debtor_val]
        
        if len(group) == 0:
            continue
        
        dem_parity    = group["predicted_dropout"].mean()
        actual_dropouts = group[group[TARGET_COL] == 1]
        eq_opportunity = (
            actual_dropouts["predicted_dropout"].mean()
            if len(actual_dropouts) > 0 else 0
        )
        actual_non_dropouts = group[group[TARGET_COL] == 0]
        fpr = (
            actual_non_dropouts["predicted_dropout"].mean()
            if len(actual_non_dropouts) > 0 else 0
        )
        
        audit_results.append({
            "group_type":        "Financial_Status",
            "group_label":       debtor_label,
            "group_size":        len(group),
            "actual_dropout_rate": round(group[TARGET_COL].mean(), 4),
            "predicted_dropout_rate": round(dem_parity, 4),
            "demographic_parity": round(dem_parity, 4),
            "equal_opportunity": round(eq_opportunity, 4),
            "false_positive_rate": round(fpr, 4),
        })
    
    # ── FAIRNESS ACROSS SCHOLARSHIP STATUS ────────────────────
    for sch_val, sch_label in [(0, "No_Scholarship"), (1, "Scholarship")]:
        group = df[df["scholarship_holder"] == sch_val]
        
        if len(group) == 0:
            continue
        
        dem_parity    = group["predicted_dropout"].mean()
        actual_dropouts = group[group[TARGET_COL] == 1]
        eq_opportunity = (
            actual_dropouts["predicted_dropout"].mean()
            if len(actual_dropouts) > 0 else 0
        )
        actual_non_dropouts = group[group[TARGET_COL] == 0]
        fpr = (
            actual_non_dropouts["predicted_dropout"].mean()
            if len(actual_non_dropouts) > 0 else 0
        )
        
        audit_results.append({
            "group_type":        "Scholarship_Status",
            "group_label":       sch_label,
            "group_size":        len(group),
            "actual_dropout_rate": round(group[TARGET_COL].mean(), 4),
            "predicted_dropout_rate": round(dem_parity, 4),
            "demographic_parity": round(dem_parity, 4),
            "equal_opportunity": round(eq_opportunity, 4),
            "false_positive_rate": round(fpr, 4),
        })
    
    audit_df = pd.DataFrame(audit_results)
    return audit_df

# ── STEP 3: DOCUMENT DISPARITY ───────────────────────────────
def document_disparity(audit_df):
    """
    We don't hide bias. We hunt it and document it honestly.
    """
    print("\n" + "="*60)
    print("FAIRNESS AUDIT REPORT — Team AryaBit")
    print("We don't hide bias. We hunt it.")
    print("="*60)
    
    for group_type in audit_df["group_type"].unique():
        group_data = audit_df[audit_df["group_type"] == group_type]
        print(f"\n📊 {group_type}:")
        print(f"{'Group':<20} {'Size':>6} {'Actual DR':>10} "
              f"{'Pred DR':>10} {'Equal Opp':>10} {'FPR':>8}")
        print("-"*70)
        
        for _, row in group_data.iterrows():
            print(
                f"{row['group_label']:<20} "
                f"{row['group_size']:>6} "
                f"{row['actual_dropout_rate']:>10.4f} "
                f"{row['predicted_dropout_rate']:>10.4f} "
                f"{row['equal_opportunity']:>10.4f} "
                f"{row['false_positive_rate']:>8.4f}"
            )
        
        # Flag disparity
        if len(group_data) == 2:
            dp_diff = abs(
                group_data.iloc[0]["demographic_parity"] -
                group_data.iloc[1]["demographic_parity"]
            )
            eo_diff = abs(
                group_data.iloc[0]["equal_opportunity"] -
                group_data.iloc[1]["equal_opportunity"]
            )
            
            if dp_diff > 0.05:
                print(f"  ⚠️  DISPARITY FOUND: Demographic Parity gap = {dp_diff:.4f}")
                print(f"      This means one group is flagged {dp_diff*100:.1f}% more often.")
                print(f"      We document this honestly — not hide it.")
            else:
                print(f"  ✅ FAIR: Demographic Parity gap = {dp_diff:.4f} (< 0.05 threshold)")
            
            if eo_diff > 0.05:
                print(f"  ⚠️  DISPARITY FOUND: Equal Opportunity gap = {eo_diff:.4f}")
            else:
                print(f"  ✅ FAIR: Equal Opportunity gap = {eo_diff:.4f} (< 0.05 threshold)")

# ── STEP 4: WRITE FAIRNESS AUDIT TO DELTA ────────────────────
def write_fairness_table(audit_df):
    logger.info("Writing Fairness Audit to Delta table...")
    
    audit_spark_df = spark.createDataFrame(audit_df).withColumn(
        "_audit_timestamp", current_timestamp()
    ).withColumn(
        "_layer", lit("gold_fairness")
    )
    
    (
        audit_spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FAIRNESS_TABLE)
    )
    logger.info(f"Fairness table written: {FAIRNESS_TABLE} ✅")

# ── STEP 5: BUILD GOLD OUTPUT TABLE ──────────────────────────
def build_gold_table(df):
    """
    Gold table: At-risk students with risk score,
    top 3 factors, intervention tier, bias flag
    """
    logger.info("Building Gold output table...")
    
    # Only flag students predicted as dropout
    at_risk = df[df["predicted_dropout"] == 1].copy()
    
    # Intervention tier based on risk score
    def get_tier(score):
        if score >= 0.75:
            return "HIGH"
        elif score >= 0.50:
            return "MEDIUM"
        else:
            return "LOW"
    
    at_risk["intervention_tier"] = at_risk["risk_score"].apply(get_tier)
    
    # Top 3 risk factors per student (rule-based on feature values)
    def get_top_factors(row):
        factors = []
        if row["overall_approval_rate"] < 0.5:
            factors.append(f"Low approval rate ({row['overall_approval_rate']:.2f})")
        if row["grade_delta"] < -1.0:
            factors.append(f"Grade declining ({row['grade_delta']:.2f})")
        if row["financial_stress_index"] >= 2:
            factors.append(f"High financial stress ({int(row['financial_stress_index'])})") 
        if row["absenteeism_trend"] < -0.2:
            factors.append(f"Worsening attendance trend ({row['absenteeism_trend']:.2f})")
        if row["avg_grade_overall"] < 8.0:
            factors.append(f"Low overall grade ({row['avg_grade_overall']:.2f})")
        # Take top 3
        return " | ".join(factors[:3]) if factors else "Academic performance decline"
    
    at_risk["top_3_risk_factors"] = at_risk.apply(get_top_factors, axis=1)
    
    # Bias flag — is this student in a high-risk demographic group?
    def get_bias_flag(row):
        if row["debtor"] == 1 and row["scholarship_holder"] == 0:
            return "REVIEW - Financial vulnerability"
        return "FAIR"
    
    at_risk["bias_flag"] = at_risk.apply(get_bias_flag, axis=1)
    
    # Create student ID (row index as proxy)
    at_risk = at_risk.reset_index()
    at_risk["student_id"] = at_risk["index"].apply(lambda x: f"STU_{x:04d}")
    
    # Select final Gold columns
    gold_pandas = at_risk[[
        "student_id", "risk_score", "intervention_tier",
        "top_3_risk_factors", "bias_flag",
        "gender", "scholarship_holder", "debtor",
        "dropout_label"
    ]]
    
    # Write to Delta
    gold_spark_df = spark.createDataFrame(gold_pandas).withColumn(
        "_gold_timestamp", current_timestamp()
    ).withColumn(
        "_layer", lit("gold")
    )
    
    (
        gold_spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_TABLE)
    )
    
    logger.info(f"Gold table written: {GOLD_TABLE} ✅")
    return gold_pandas

# ── STEP 6: DISPLAY GOLD TABLE ───────────────────────────────
def display_gold_table():
    print("\n" + "="*60)
    print("GOLD TABLE — AT RISK STUDENTS")
    print("The Gold table doesn't just show who's at risk.")
    print("It proves the risk score was earned — not assigned.")
    print("="*60)
    
    spark.sql(f"""
        SELECT
            student_id,
            ROUND(risk_score, 4) as risk_score,
            intervention_tier,
            top_3_risk_factors,
            bias_flag
        FROM {GOLD_TABLE}
        ORDER BY risk_score DESC
        LIMIT 10
    """).show(truncate=False)
    
    print("\nINTERVENTION TIER DISTRIBUTION:")
    spark.sql(f"""
        SELECT
            intervention_tier,
            COUNT(*) as count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
        FROM {GOLD_TABLE}
        GROUP BY intervention_tier
        ORDER BY count DESC
    """).show()
    
    print("\nTOTAL AT-RISK STUDENTS FLAGGED:")
    count = spark.sql(f"SELECT COUNT(*) as total FROM {GOLD_TABLE}").collect()[0][0]
    print(f"Total: {count}")

# ── MAIN EXECUTION ───────────────────────────────────────────
def run_fairness_pipeline():
    logger.info("Starting Fairness Audit Pipeline — AryaBit")
    try:
        # Generate predictions
        df_with_predictions = load_and_predict()
        
        # Compute fairness metrics
        audit_df = compute_fairness_metrics(df_with_predictions)
        
        # Document findings
        document_disparity(audit_df)
        
        # Write fairness audit table
        write_fairness_table(audit_df)
        
        # Build Gold table
        gold_df = build_gold_table(df_with_predictions)
        
        # Display results
        display_gold_table()
        
        logger.info("Fairness Pipeline COMPLETE ✅")
        return True
        
    except Exception as e:
        logger.error(f"Fairness Pipeline FAILED: {str(e)}")
        raise e

# ── RUN ──────────────────────────────────────────────────────
run_fairness_pipeline()